# Phase 2 — Hybrid Fusion & Re-ranking (per subset)

In [18]:

# Configuration
from pathlib import Path
import os, json, math, re, numpy as np, pandas as pd
from typing import Dict, List, Tuple
from tqdm import tqdm

WORK_DIR = Path("./work")
DATASET_KEY = "beir_trec-covid"   # e.g., "beir_trec-covid" or "msmarco-passage_train"
SUBSET_ID = "subset_3"            # change to subset_2, subset_3, etc.
SUBSET_DIR = WORK_DIR / "subsets" / DATASET_KEY / SUBSET_ID

TOP_K_RUN = 1000
K_EVAL = 10
RANDOM_SEED = 13

print("Using subset:", SUBSET_DIR)
assert SUBSET_DIR.exists(), f"Subset folder not found: {SUBSET_DIR}"


Using subset: work/subsets/beir_trec-covid/subset_3


In [19]:

# IO helpers
def read_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)

def write_trec(path: Path, records, system_name="sys"):
    per_q = {}
    for qid, docid, score in records:
        per_q.setdefault(qid, []).append((docid, float(score)))
    with path.open("w", encoding="utf-8") as f:
        for qid, pairs in per_q.items():
            pairs.sort(key=lambda x: x[1], reverse=True)
            for rank, (docid, score) in enumerate(pairs[:TOP_K_RUN], start=1):
                f.write(f"{qid} Q0 {docid} {rank} {score:.6f} {system_name}\n")

def read_trec(path: Path) -> Dict[str, List[Tuple[str, float]]]:
    per_q = {}
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 6: 
                continue
            qid, _, docid, _, score, _sys = parts[:6]
            per_q.setdefault(qid, []).append((docid, float(score)))
    for qid in per_q:
        per_q[qid].sort(key=lambda x: x[1], reverse=True)
    return per_q


In [20]:

# Load corpus, queries, qrels
corpus = list(read_jsonl(SUBSET_DIR / "corpus.jsonl"))
queries = list(read_jsonl(SUBSET_DIR / "queries.jsonl"))

qrels = {}
for r in read_jsonl(SUBSET_DIR / "qrels.jsonl"):
    qrels.setdefault(str(r["qid"]), {})[str(r["doc_id"])] = int(r["rel"])

doc_text = {r["doc_id"]: r["text"] for r in corpus}
qid_text = {r["qid"]: r["text"] for r in queries}

print("Loaded:", len(corpus), "docs;", len(queries), "queries;", "qrels entries:", sum(len(v) for v in qrels.values()))


Loaded: 55930 docs; 16 queries; qrels entries: 7825


In [23]:

# Try to load BM25 and Dense runs; rebuild if missing
bm25_run_path  = SUBSET_DIR / "run_bm25.trec"
dense_run_path = SUBSET_DIR / "run_dense.trec"

def ensure_bm25_run():
    if bm25_run_path.exists():
        return
    print("BM25 run missing -> building a quick BM25 baseline (Rank-BM25).")
    from rank_bm25 import BM25Okapi
    import re, numpy as np
    def tok(s): return re.findall(r"[a-z0-9]+", s.lower())
    docs = [doc_text[d] for d in doc_text.keys()]
    docids = list(doc_text.keys())
    bm25 = BM25Okapi([tok(x) for x in docs])
    recs=[]
    for q in tqdm(queries, desc="BM25 retrieve"):
        scores = bm25.get_scores(tok(q["text"]))
        idxs = np.argsort(scores)[::-1][:TOP_K_RUN]
        for rank, i in enumerate(idxs, start=1):
            recs.append((q["qid"], docids[i], float(scores[i])))
    write_trec(bm25_run_path, recs, "bm25")

def ensure_dense_run():
    if dense_run_path.exists():
        return
    print("Dense run missing -> building a quick dense baseline (MiniLM + FAISS).")
    import faiss, torch, re
    from sentence_transformers import SentenceTransformer
    def tok(s): return re.findall(r"[a-z0-9]+", s.lower())
    # encode docs
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda" if torch.cuda.is_available() else "cpu")
    model.max_seq_length = 256
    texts = [doc_text[d] for d in doc_text.keys()]
    xb = model.encode(texts, batch_size=64, show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True)
    dim = xb.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(xb)
    docids = list(doc_text.keys())
    # encode queries
    qtexts = [qid_text[q["qid"]] for q in queries]
    xq = model.encode(qtexts, batch_size=64, show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True)
    D, I = index.search(xq, TOP_K_RUN)
    recs=[]
    for qi, q in enumerate(queries):
        for rank, j in enumerate(I[qi], start=1):
            recs.append((q["qid"], docids[j], float(D[qi, rank-1])))
    write_trec(dense_run_path, recs, "dense")

ensure_bm25_run()
ensure_dense_run()

bm25 = read_trec(bm25_run_path)
dense = read_trec(dense_run_path)

print("Loaded runs:", bm25_run_path.name, "and", dense_run_path.name)
print("Queries:", len(bm25), len(dense))


Loaded runs: run_bm25.trec and run_dense.trec
Queries: 16 16


In [24]:

# Score normalization utilities
import numpy as np

def minmax_norm(scores):
    vals = np.array([s for _, s in scores], dtype=float)
    if len(vals)==0: return {}
    lo, hi = float(np.min(vals)), float(np.max(vals))
    if hi <= lo: return {doc: 0.0 for doc,_ in scores}
    return {doc: (s - lo) / (hi - lo) for doc, s in scores}

def zscore_norm(scores):
    vals = np.array([s for _, s in scores], dtype=float)
    if len(vals)==0: return {}
    mu, sigma = float(np.mean(vals)), float(np.std(vals) + 1e-9)
    return {doc: (s - mu) / sigma for doc, s in scores}

def rank_norm(scores):
    n = len(scores)
    return {doc: (n - r) / max(1, n-1) for r, (doc, _) in enumerate(sorted(scores, key=lambda x: x[1], reverse=True), start=1)}


In [25]:

# Hybrid fusion
def fuse_per_query(qid, s_bm25, s_dense, norm="minmax", alpha=0.5, topk=TOP_K_RUN):
    b = s_bm25.get(qid, [])
    d = s_dense.get(qid, [])
    if norm == "minmax":
        bn = minmax_norm(b); dn = minmax_norm(d)
    elif norm == "zscore":
        bn = zscore_norm(b); dn = zscore_norm(d)
    elif norm == "rank":
        bn = rank_norm(b); dn = rank_norm(d)
    else:
        raise ValueError("Unknown norm")
    cand = set([doc for doc,_ in b]) | set([doc for doc,_ in d])
    fused=[]
    for doc in cand:
        sb = bn.get(doc, 0.0); sd = dn.get(doc, 0.0)
        sc = alpha*sb + (1.0-alpha)*sd
        fused.append((doc, sc))
    fused.sort(key=lambda x: x[1], reverse=True)
    return fused[:topk]

def build_hybrid_run(bm25, dense, norm: str, alpha: float):
    recs=[]
    for qid in qid_text.keys():
        fused = fuse_per_query(qid, bm25, dense, norm=norm, alpha=alpha, topk=TOP_K_RUN)
        for rank,(doc,score) in enumerate(fused, start=1):
            recs.append((qid, doc, float(score)))
    return recs


In [26]:

# Grid-search alpha & normalization
norms = ["minmax", "zscore", "rank"]
alphas = [round(x,2) for x in np.linspace(0.0, 1.0, 11)]
rows = []

for nm in norms:
    for a in alphas:
        recs = build_hybrid_run(bm25, dense, nm, a)
        out_path = SUBSET_DIR / f"run_hybrid_{nm}_a{a:.2f}.trec"
        write_trec(out_path, recs, system_name=f"hybrid_{nm}_{a:.2f}")
        run = read_trec(out_path)
        #n10 = ndcg_at_k(run, qrels, k=K_EVAL, skip_no_rels=True)
        #m10 = mrr_at_k(run, qrels, k=K_EVAL)
        #r100 = recall_at_k(run, qrels, k=100)
        #rows.append({"norm": nm, "alpha": a, "nDCG@10": n10, "MRR@10": m10, "Recall@100": r100, "run_path": str(out_path)})

import pandas as pd
#df = pd.DataFrame(rows).sort_values(["nDCG@10","MRR@10"], ascending=False)
#df.head(10)
